# Clases y objetos · Python moderno

`match`/`case`, anotaciones de tipos con `typing`, `dataclasses` a fondo y gestores de contexto.

## 1. `match` con literales

Escribir una función que reciba un código de estado HTTP y devuelva su categoría usando `match`/`case` con literales y alternativas `|`.

In [1]:
def describe_estado(codigo):
    match codigo:
        case 200 | 201 | 204:
            return "éxito"
        case 301 | 302:
            return "redirección"
        case 404:
            return "no encontrado"
        case 500 | 503:
            return "error del servidor"
        case _:
            return "desconocido"

for codigo in (200, 302, 404, 503, 418):
    print(codigo, describe_estado(codigo))
assert describe_estado(204) == "éxito"

200 éxito
302 redirección
404 no encontrado
503 error del servidor
418 desconocido


## 2. `match` con secuencias

Escribir un intérprete de órdenes de un juego de texto (`ir norte`, `coger llave mapa`, `mover 3 4`, `salir`) usando patrones de secuencia.

In [2]:
def ejecuta(orden):
    match orden.split():
        case []:
            return "no has escrito nada"
        case ["salir"]:
            return "adiós"
        case ["ir", direccion]:
            return f"vas hacia el {direccion}"
        case ["coger", *objetos] if objetos:
            return f"coges: {', '.join(objetos)}"
        case ["mover", x, y]:
            return f"te mueves a ({int(x)}, {int(y)})"
        case _:
            return f"no entiendo «{orden}»"

for orden in ("ir norte", "coger llave mapa", "coger", "mover 3 4", "", "salir", "bailar"):
    print(f"{orden!r:20} → {ejecuta(orden)}")
assert ejecuta("coger llave mapa") == "coges: llave, mapa"

'ir norte'           → vas hacia el norte
'coger llave mapa'   → coges: llave, mapa
'coger'              → no entiendo «coger»
'mover 3 4'          → te mueves a (3, 4)
''                   → no has escrito nada
'salir'              → adiós
'bailar'             → no entiendo «bailar»


## 3. `match` con mapeos

Escribir una función que procese eventos representados como diccionarios (`click`, `tecla`, `scroll`) usando patrones de mapeo y de tipo.

In [3]:
def procesa_evento(evento):
    match evento:
        case {"tipo": "click", "x": int(x), "y": int(y)}:
            return f"click en ({x}, {y})"
        case {"tipo": "tecla", "tecla": str(tecla), **resto}:
            modificadores = [m for m, activo in resto.items() if activo]
            return "tecla " + "+".join([*modificadores, tecla])
        case {"tipo": "scroll", "delta": delta}:
            return "scroll " + ("arriba" if delta > 0 else "abajo")
        case {"tipo": tipo}:
            return f"evento {tipo!r} no soportado"
        case _:
            return "evento no válido"

eventos = [
    {"tipo": "click", "x": 10, "y": 20},
    {"tipo": "tecla", "tecla": "s", "ctrl": True, "shift": False},
    {"tipo": "scroll", "delta": -3},
    {"tipo": "toque"},
    {"x": 1},
]
for evento in eventos:
    print(procesa_evento(evento))
assert procesa_evento(eventos[1]) == "tecla ctrl+s"

click en (10, 20)
tecla ctrl+s
scroll abajo
evento 'toque' no soportado
evento no válido


## 4. `match` con clases y guardas

Definir las clases `Punto`, `Circulo` y `Rectangulo` y escribir una función que describa cada figura usando patrones de clase, patrones anidados y guardas `if`.

In [4]:
import math
from dataclasses import dataclass

@dataclass
class Punto:
    x: float
    y: float

@dataclass
class Circulo:
    centro: Punto
    radio: float

@dataclass
class Rectangulo:
    esquina: Punto
    ancho: float
    alto: float

def describe(figura):
    match figura:
        case Punto(x=0, y=0):
            return "punto en el origen"
        case Punto(x=0) | Punto(y=0):
            return "punto sobre un eje"
        case Punto(x, y):
            return f"punto ({x}, {y})"
        case Circulo(radio=r) if r <= 0:
            return "círculo no válido"
        case Circulo(Punto(0, 0), r):
            return f"círculo de radio {r} centrado en el origen"
        case Circulo(_, r):
            return f"círculo de área {math.pi * r ** 2:.2f}"
        case Rectangulo(_, ancho, alto) if ancho == alto:
            return f"cuadrado de lado {ancho}"
        case Rectangulo(_, ancho, alto):
            return f"rectángulo de {ancho} × {alto}"
        case _:
            return "figura desconocida"

figuras = [Punto(0, 0), Punto(0, 5), Punto(2, 3), Circulo(Punto(0, 0), 2), Circulo(Punto(1, 1), 1),
           Circulo(Punto(1, 1), -1), Rectangulo(Punto(0, 0), 4, 4), Rectangulo(Punto(0, 0), 2, 5), "triángulo"]
for figura in figuras:
    print(describe(figura))

punto en el origen
punto sobre un eje
punto (2, 3)
círculo de radio 2 centrado en el origen
círculo de área 3.14
círculo no válido
cuadrado de lado 4
rectángulo de 2 × 5
figura desconocida


## 5. Anotaciones de tipos básicas

Escribir funciones anotadas con tipos de colecciones (`list[str]`), valores opcionales (`Optional[str]` y `int | None`) y un alias de tipo. Comprobar que las anotaciones no se verifican al ejecutar.

In [5]:
from typing import Optional, get_type_hints

type Matriz = list[list[float]]

def busca_indice(elementos: list[str], objetivo: str) -> int | None:
    for i, elemento in enumerate(elementos):
        if elemento == objetivo:
            return i
    return None

def saluda(nombre: Optional[str] = None) -> str:
    return f"hola {nombre or 'desconocido'}"

def traspuesta(matriz: Matriz) -> Matriz:
    return [list(fila) for fila in zip(*matriz)]

print(busca_indice(["pan", "leche", "huevos"], "leche"), busca_indice(["pan"], "queso"))
print(saluda(), saluda("Carmen"))
print(traspuesta([[1, 2, 3], [4, 5, 6]]))
print(get_type_hints(busca_indice))
print(saluda(42))  # las anotaciones no impiden pasar otro tipo

1 None
hola desconocido hola Carmen
[[1, 4], [2, 5], [3, 6]]
{'elementos': list[str], 'objetivo': <class 'str'>, 'return': int | None}
hola 42


## 6. Genéricos

Crear una clase genérica `Pila[T]` con `apila`, `desapila` y `cima`, y una función genérica `primero[T]` que devuelva el primer elemento de una lista o un valor por defecto.

In [6]:
class Pila[T]:
    def __init__(self) -> None:
        self._elementos: list[T] = []

    def apila(self, elemento: T) -> None:
        self._elementos.append(elemento)

    def desapila(self) -> T:
        if not self._elementos:
            raise IndexError("la pila está vacía")
        return self._elementos.pop()

    def cima(self) -> T | None:
        return self._elementos[-1] if self._elementos else None

    def __len__(self) -> int:
        return len(self._elementos)

def primero[T](elementos: list[T], defecto: T | None = None) -> T | None:
    return elementos[0] if elementos else defecto

pila: Pila[int] = Pila()
for n in (1, 2, 3):
    pila.apila(n)
print(pila.desapila(), pila.cima(), len(pila))

print(primero(["a", "b"]), primero([], defecto="vacía"))
print(Pila.__type_params__, primero.__type_params__)
assert pila.desapila() == 2 and pila.desapila() == 1 and pila.cima() is None

3 2 2
a vacía
(T,) (T,)


## 7. `TypedDict` y `Callable`

Definir un `TypedDict` `Producto` con una clave opcional y escribir una función que aplique a los precios de una lista de productos cualquier operación de tipo `Callable[[float], float]`: añadir el IGIC (7 %) o aplicar un descuento.

In [7]:
from typing import Callable, NotRequired, TypedDict

class Producto(TypedDict):
    nombre: str
    precio: float
    oferta: NotRequired[bool]

def aplica(productos: list[Producto], operacion: Callable[[float], float]) -> list[Producto]:
    return [{**p, "precio": round(operacion(p["precio"]), 2)} for p in productos]

def con_igic(precio: float) -> float:
    return precio * 1.07

def con_descuento(porcentaje: float) -> Callable[[float], float]:
    return lambda precio: precio * (1 - porcentaje / 100)

productos: list[Producto] = [
    {"nombre": "queso", "precio": 12.5},
    {"nombre": "mojo", "precio": 3.2, "oferta": True},
]
print(aplica(productos, con_igic))
print(aplica(productos, con_descuento(20)))
print(Producto.__required_keys__, Producto.__optional_keys__)
assert aplica(productos, con_descuento(20))[0]["precio"] == 10.0

[{'nombre': 'queso', 'precio': 13.38}, {'nombre': 'mojo', 'precio': 3.42, 'oferta': True}]
[{'nombre': 'queso', 'precio': 10.0}, {'nombre': 'mojo', 'precio': 2.56, 'oferta': True}]
frozenset({'nombre', 'precio'}) frozenset({'oferta'})


## 8. `Protocol`

Definir un protocolo `ConArea` con un método `area` y escribir una función que sume el área de cualquier objeto que lo cumpla, sin necesidad de herencia. Comprobarlo en tiempo de ejecución con `runtime_checkable`.

In [8]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class ConArea(Protocol):
    def area(self) -> float: ...

class Cuadrado:
    def __init__(self, lado: float) -> None:
        self.lado = lado

    def area(self) -> float:
        return self.lado ** 2

class Triangulo:
    def __init__(self, base: float, altura: float) -> None:
        self.base, self.altura = base, altura

    def area(self) -> float:
        return self.base * self.altura / 2

def area_total(figuras: list[ConArea]) -> float:
    return sum(f.area() for f in figuras)

figuras = [Cuadrado(3), Triangulo(4, 5), Cuadrado(1)]
print(area_total(figuras))
print(isinstance(Cuadrado(2), ConArea), isinstance("hola", ConArea))
assert area_total(figuras) == 20

20.0
True False


## 9. `dataclasses`: `field`, `__post_init__` y `order`

Crear una dataclass `Alumno` ordenable por su nota media, con una lista de notas creada con `default_factory`, una media calculada en `__post_init__` (que además valide que las notas estén entre 0 y 10) y campos excluidos de la comparación.

In [9]:
from dataclasses import asdict, dataclass, field

@dataclass(order=True)
class Alumno:
    media: float = field(init=False)
    nombre: str = field(compare=False)
    notas: list[float] = field(default_factory=list, compare=False, repr=False)

    def __post_init__(self):
        if not all(0 <= n <= 10 for n in self.notas):
            raise ValueError(f"{self.nombre}: notas fuera de rango")
        self.media = round(sum(self.notas) / len(self.notas), 2) if self.notas else 0.0

clase = [Alumno("Ana", [8, 9, 7.5]), Alumno("Luis", [5, 6]), Alumno("Marta", [9.5, 10]), Alumno("Nuevo")]
for alumno in sorted(clase, reverse=True):
    print(alumno)

print(asdict(max(clase)))
try:
    Alumno("Pepe", [11, 4])
except ValueError as e:
    print("ValueError:", e)
assert max(clase).nombre == "Marta"
assert Alumno("A").notas is not Alumno("B").notas

Alumno(media=9.75, nombre='Marta')
Alumno(media=8.17, nombre='Ana')
Alumno(media=5.5, nombre='Luis')
Alumno(media=0.0, nombre='Nuevo')
{'media': 9.75, 'nombre': 'Marta', 'notas': [9.5, 10]}
ValueError: Pepe: notas fuera de rango


## 10. `dataclasses`: `frozen` y `slots`

Crear una dataclass inmutable `Coordenada` (`frozen=True`, `slots=True`) que valide la latitud y la longitud. Comprobar que no se puede modificar, que puede usarse en un conjunto, que `replace` crea una copia modificada y que con `slots` no se pueden añadir atributos nuevos.

In [10]:
from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True, slots=True)
class Coordenada:
    latitud: float
    longitud: float

    def __post_init__(self):
        if not (-90 <= self.latitud <= 90 and -180 <= self.longitud <= 180):
            raise ValueError(f"coordenada fuera de rango: {self.latitud}, {self.longitud}")

las_palmas = Coordenada(28.12, -15.43)
try:
    las_palmas.latitud = 0
except FrozenInstanceError as e:
    print("FrozenInstanceError:", e)

visitadas = {las_palmas, Coordenada(28.47, -16.25), Coordenada(28.12, -15.43)}
print(len(visitadas), "lugares distintos")
print(replace(las_palmas, longitud=-15.40))
try:
    Coordenada(100, 0)
except ValueError as e:
    print("ValueError:", e)
assert len(visitadas) == 2

FrozenInstanceError: cannot assign to field 'latitud'
2 lugares distintos
Coordenada(latitud=28.12, longitud=-15.4)
ValueError: coordenada fuera de rango: 100, 0


In [11]:
@dataclass(slots=True)
class PuntoLigero:
    x: int
    y: int

p = PuntoLigero(1, 2)
print(PuntoLigero.__slots__, hasattr(p, "__dict__"))
try:
    p.z = 3
except AttributeError as e:
    print("AttributeError:", e)

('x', 'y') False
AttributeError: 'PuntoLigero' object has no attribute 'z' and no __dict__ for setting new attributes


## 11. Gestor de contexto con una clase

Crear una clase `Transaccion` con `__enter__` y `__exit__` que permita modificar un diccionario y, si dentro del bloque `with` se produce una excepción, deshaga los cambios sin silenciar la excepción.

In [12]:
class Transaccion:
    def __init__(self, datos):
        self.datos = datos

    def __enter__(self):
        self.copia = dict(self.datos)
        return self.datos

    def __exit__(self, tipo, valor, traza):
        if tipo is not None:
            self.datos.clear()
            self.datos.update(self.copia)
            print(f"cambios deshechos por {tipo.__name__}: {valor}")
        return False  # no silencia la excepción

inventario = {"manzanas": 10, "peras": 5}

with Transaccion(inventario) as inv:
    inv["manzanas"] -= 3
print(inventario)

try:
    with Transaccion(inventario) as inv:
        inv["peras"] -= 2
        inv["kiwis"] -= 1
except KeyError as e:
    print("KeyError capturado fuera del with:", e)
print(inventario)
assert inventario == {"manzanas": 7, "peras": 5}

{'manzanas': 7, 'peras': 5}
cambios deshechos por KeyError: 'kiwis'
KeyError capturado fuera del with: 'kiwis'
{'manzanas': 7, 'peras': 5}


## 12. `contextlib`: `contextmanager` y `suppress`

Escribir con `contextlib.contextmanager` un gestor `fichero_temporal` que cree un fichero, lo entregue al bloque `with` y lo borre al salir, y otro `temporizador` que mida el tiempo del bloque. Usar `suppress` para ignorar el error al borrar un fichero que no existe.

In [13]:
import time
from contextlib import contextmanager, suppress
from pathlib import Path

@contextmanager
def fichero_temporal(nombre, contenido):
    ruta = Path(nombre)
    ruta.write_text(contenido, encoding="utf-8")
    try:
        yield ruta
    finally:
        ruta.unlink()

@contextmanager
def temporizador(etiqueta):
    inicio = time.perf_counter()
    try:
        yield
    finally:
        print(f"{etiqueta}: {time.perf_counter() - inicio:.4f} s")

with temporizador("lectura"), fichero_temporal("temporal.txt", "línea 1\nlínea 2\n") as ruta:
    print(ruta.read_text(encoding="utf-8").splitlines(), "existe:", ruta.exists())
print("existe después:", ruta.exists())

with suppress(FileNotFoundError):
    Path("no-existe.txt").unlink()
print("borrado ignorado sin error")
assert not ruta.exists()

['línea 1', 'línea 2'] existe: True
lectura: 0.0020 s
existe después: False
borrado ignorado sin error
